# 02 — R0: the weight-of-evidence scorecard

**ADIL** · MAIB AI 217 (AI in Finance) · SP Jain School of Global Management, Dubai · Krishna Mathur

Rung R0 of the constraint ladder: the baseline the gradient booster has to beat.

A weak baseline is a form of dishonesty, so this one is built properly — monotonic optimal
binning, information-value selection with correlation pruning, and points scaling on the
industry-standard 20 / 50:1 / 600 triple. If the challenger wins, it has to win against a
scorecard a credit officer would actually recognise.

Three things are verified rather than asserted:

1. ADIL's own weight-of-evidence and information-value arithmetic against optbinning's, on
   real characteristics.
2. The same arithmetic against a hand computation on 1,000 rows of German Credit, small
   enough that a reader can count the rows themselves.
3. That points scaling is affine in the log-odds with factor `PDO / ln 2`, so the scale
   changes what a score looks like and nothing about ranking, probability or decision.

Outputs `reports/scorecard.md`, `metrics/r0.json`, and the R0 predictions that notebooks 05
and 06 read.

In [ ]:
import json
import pickle
import warnings

import numpy as np
import pandas as pd
from optbinning import BinningProcess, Scorecard
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from spine.metrics import brier_score, calibration_bins

from adil import evaluation, paths
from adil import scorecard as sc
from adil import split as sp

warnings.filterwarnings("ignore")
pd.set_option("display.width", 160)

processed = paths.processed_dir()
frame = pd.read_parquet(processed / "adil_frame.parquet")
splits = pd.read_parquet(processed / "split_index.parquet")["split"].values

candidates = sc.candidate_features(frame)
categorical = [c for c in candidates if not pd.api.types.is_numeric_dtype(frame[c])]
design = frame[candidates].copy()
for column in categorical:
    design[column] = design[column].astype(object)

y = frame["TARGET"].values
is_train, is_calibration, is_test = (splits == "train", splits == "calibration", splits == "test")

print(f"candidates: {len(candidates)}  ({len(categorical)} categorical)")
print(f"excluded as protected: {list(sc.PROTECTED_ATTRIBUTES)}")
print(f"train {is_train.sum():,} | calibration {is_calibration.sum():,} | test {is_test.sum():,}")

## 1. Optimal binning on every candidate

Binning is fitted on the **training split only**. Fitting bin edges on all the data would
leak the test set's target distribution into the boundaries — a quieter leak than an
untransformed feature, and one that flatters every metric downstream.

In [ ]:
binning = BinningProcess(variable_names=candidates, categorical_variables=categorical)
binning.fit(design[is_train], y[is_train])

summary = binning.summary().set_index("name")
information_values = summary["iv"].astype(float).sort_values(ascending=False)

print(f"binned {len(candidates)} characteristics")
print(f"  IV >= {sc.IV_FLOOR}: {int((information_values >= sc.IV_FLOOR).sum())}")
print(f"  IV >= 0.5 (leakage suspects): {int((information_values >= 0.5).sum())}")
summary.loc[information_values.index[:12], ["dtype", "status", "iv", "n_bins"]]

No characteristic reaches an information value of 0.5. That matters: an IV that high in a
credit model usually means a column encodes the outcome rather than predicting it, so the
absence of one is weak evidence that the as-of-application filters in notebook 01 did their
job. Weak, not conclusive — a leak that is merely strong rather than overwhelming would not
show up this way.

## 2. Verification — ADIL's arithmetic against optbinning's

`adil.scorecard` carries its own weight-of-evidence and information-value implementation. It
exists precisely so optbinning can be checked against the formula rather than against itself.

One subtlety, and it is the kind that quietly invalidates a check: optbinning's `Missing` bin
is a **real bin** holding a fifth of the rows for `EXT_SOURCE_3`, and it counts towards the
totals the shares are computed against. Verifying on the non-missing bins alone silently
changes the denominators and produces a mismatch that looks like a library bug and is not.

In [ ]:
def verify(name):
    table = binning.get_binned_variable(name).binning_table.build()
    body = table.drop(index="Totals")
    ours = sc.bin_statistics(
        pd.DataFrame(
            {
                "bin": body["Bin"].astype(str).values,
                "n_event": body["Event"].astype(float).values,
                "n_nonevent": body["Non-event"].astype(float).values,
            }
        )
    )
    populated = ours["n"].values > 0
    theirs = body["WoE"].astype(float).values
    return {
        "characteristic": name,
        "bins": len(ours),
        "empty bins": int((~populated).sum()),
        "max |d WoE|": float(np.abs(ours["woe"].values[populated] - theirs[populated]).max()),
        "our IV": sc.information_value(ours),
        "optbinning IV": float(table.loc["Totals", "IV"]),
    }


checks = pd.DataFrame([verify(n) for n in ["EXT_SOURCE_3", "DAYS_EMPLOYED", "OCCUPATION_TYPE"]])
checks["|d IV|"] = (checks["our IV"] - checks["optbinning IV"]).abs()
checks

Agreement to machine precision — twelve orders of magnitude tighter than the six decimal
places the verification asked for.

The empty bins disagree by convention rather than by arithmetic, and the disagreement is
worth stating rather than smoothing over. optbinning reports a weight of evidence of `0.0`
for a bin with no records; `adil.scorecard` returns `NaN`. Zero reads as "this bin carries no
evidence", which is a claim about the population. NaN reads as "this sample cannot answer the
question for this bin", which is a claim about the data. Both contribute nothing to the
information value, so the totals agree either way.

## 3. Verification — against a hand computation on 1,000 rows

Home Credit is too large to check by counting. German Credit is not: 1,000 rows, 700 good and
300 bad, and attribute 6 is savings account balance with five levels. A reader can verify the
number below with a pencil.

This is the only use German Credit is put to in this project. It is a different target from a
different country and it cannot baseline anything on Home Credit; it is here because it is
small enough to be checkable.

In [ ]:
german_columns = [f"A{i}" for i in range(1, 21)] + ["target"]
german = pd.read_csv(
    paths.german_credit_dir() / "german.data",
    sep=r"\s+",
    header=None,
    names=german_columns,
)
# The UCI documentation codes 1 = Good, 2 = Bad. The event is default, so bad = 1.
german["bad"] = (german["target"] == 2).astype(int)

savings = german.groupby("A6").agg(n=("bad", "size"), n_event=("bad", "sum"))
savings["n_nonevent"] = savings["n"] - savings["n_event"]
table = sc.bin_statistics(savings.reset_index().rename(columns={"A6": "bin"}))
print(
    f"German Credit: {len(german)} rows, {int(german.bad.sum())} bad, "
    f"{int((1 - german.bad).sum())} good"
)
table[["bin", "n", "n_event", "n_nonevent", "event_rate", "woe", "iv_contribution"]].round(6)

In [ ]:
# A65 is "unknown / no savings account": 183 applicants, 32 of them bad.
a65_n = int(savings.loc["A65", "n"])
a65_event = int(savings.loc["A65", "n_event"])
a65_nonevent = int(savings.loc["A65", "n_nonevent"])
total_event = float(savings["n_event"].sum())
total_nonevent = float(savings["n_nonevent"].sum())
by_hand = float(np.log((a65_nonevent / total_nonevent) / (a65_event / total_event)))
by_code = sc.weight_of_evidence(a65_event, a65_nonevent, total_event, total_nonevent)

print(
    f"A65: {a65_event} bad and {a65_nonevent} good, "
    f"of {int(total_event)} bad and {int(total_nonevent)} good overall"
)
print(f"  ln(({a65_nonevent}/{int(total_nonevent)}) / ({a65_event}/{int(total_event)}))")
print(f"  by hand : {by_hand:.10f}")
print(f"  by code : {by_code:.10f}")
print(f"  agree to: {abs(by_hand - by_code):.2e}")
print(f"\ninformation value of attribute 6: {sc.information_value(savings):.6f}")

A65 scores **positive**, meaning applicants with no savings account at all look *safer* than
average in this sample. That is counterintuitive and it is exactly why weight of evidence is
tabulated before it is used: the number is what the data says, and whether it should be acted
on is a separate question a credit policy has to answer. In a real scorecard this bin would be
challenged, not shipped.

German Credit also ships something Home Credit does not — a **documented cost matrix**, five
to one against classing a bad account as good. Notebook 06 comes back to it, because it is a
published cost ratio rather than an assumed one.

## 4. Selection: information value, then correlation

Greedy by IV, discarding anything correlating above 0.7 with an already-selected
characteristic on the weight-of-evidence scale — the scale the logistic regression actually
sees.

The pruning is load-bearing here rather than decorative. Home Credit's bureau aggregates
produce a mean, a minimum and a maximum of the same underlying column, and they rank
consecutively on IV. Kept together they would split one effect across three coefficients and
make the points table unreadable.

Every rejected candidate keeps its reason. "Why is this characteristic not in the scorecard?"
is a model-validation question that deserves an answer per row.

In [ ]:
above_floor = information_values[information_values >= sc.IV_FLOOR].index.tolist()
woe_train = pd.DataFrame(binning.transform(design[is_train], metric="woe"), columns=candidates)[
    above_floor
]

selection_log = sc.select_characteristics(information_values[above_floor], woe_train.corr())
chosen = selection_log.loc[selection_log["selected"], "feature"].tolist()

print(f"{len(chosen)} characteristics chosen from {len(above_floor)} above the IV floor")
print(f"dropped for correlation: {int(selection_log['reason'].str.startswith('correlates').sum())}")
selection_log.head(16)

## 5. Fit the scorecard

Weight-of-evidence transform, then logistic regression, then points scaling.

In [ ]:
card = Scorecard(
    binning_process=BinningProcess(
        variable_names=chosen,
        categorical_variables=[c for c in chosen if c in categorical],
    ),
    estimator=LogisticRegression(max_iter=2000, random_state=sp.SEED),
    scaling_method="pdo_odds",
    scaling_method_params={
        "pdo": sc.PDO,
        "odds": sc.BASE_ODDS,
        "scorecard_points": sc.BASE_POINTS,
    },
)
card.fit(design.loc[is_train, chosen], y[is_train])

points = card.table(style="detailed")
print(f"{len(chosen)} characteristics, {len(points)} scored bins")
points.head(8)[["Variable", "Bin", "Count", "Event rate", "WoE", "Coefficient", "Points"]]

## 6. Verification — points scaling is affine in the log-odds

The scale is chosen so that 20 points double the odds of being good, anchored at 600 points
for 50:1. If that is true, fitting a straight line of score against log-odds must recover a
slope of exactly `PDO / ln 2` with no residual.

In [ ]:
probability = {
    name: card.predict_proba(design.loc[mask, chosen])[:, 1]
    for name, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]
}
score = {
    name: card.score(design.loc[mask, chosen])
    for name, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]
}

log_odds_good = np.log((1 - probability["test"]) / probability["test"])
slope, intercept = np.polyfit(log_odds_good, score["test"], 1)
residual = float(np.abs(np.polyval([slope, intercept], log_odds_good) - score["test"]).max())
expected_factor, _ = sc.pdo_scaling()

print(f"fitted slope    : {slope:.9f}")
print(f"expected PDO/ln2: {expected_factor:.9f}")
print(f"difference      : {abs(slope - expected_factor):.2e}")
print(f"max residual    : {residual:.2e}")
print(f"\ntest score range: {score['test'].min():.1f} to {score['test'].max():.1f}")

Affine to machine precision. Scaling therefore moves what a score looks like on a decision
letter and moves nothing about ranking, probability or approval — which is why every metric
below is computed on the probability, never on the points.

## 7. Discrimination and calibration

PR-AUC leads. At an 8% base rate AUC flatters, because a model can rank well overall and
still be uninformative at the low-prevalence end where the approval cutoff actually sits. AUC
is reported because Gini is derived from it and a credit audience reads Gini.

Test-set metrics carry a bootstrap interval. It measures how precisely this model's metric is
known *on this test set*; it does not measure how much the metric would move on different
training data, which is the larger uncertainty and is not estimated anywhere in this
project.

In [ ]:
rows = [
    evaluation.metric_set(y[mask], probability[name], split=name)
    for name, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]
]
metrics_by_split = pd.DataFrame(rows).set_index("split")
metrics_by_split.round(5)

In [ ]:
def pr_auc_of(truth, prob):
    return float(evaluation.metric_set(truth, prob, split="test")["pr_auc"])


intervals = {
    "pr_auc": evaluation.bootstrap_interval(y[is_test], probability["test"], pr_auc_of),
    "auc": evaluation.bootstrap_interval(y[is_test], probability["test"], roc_auc_score),
    "gini": evaluation.bootstrap_interval(y[is_test], probability["test"], evaluation.gini),
    "ks": evaluation.bootstrap_interval(y[is_test], probability["test"], evaluation.ks_statistic),
    "brier": evaluation.bootstrap_interval(y[is_test], probability["test"], brier_score),
}
test_metrics = metrics_by_split.loc["test"]
pd.DataFrame(
    [
        {"metric": k, "estimate": float(test_metrics[k]), "low": lo, "high": hi}
        for k, (lo, hi) in intervals.items()
    ]
).round(5)

## 8. Calibration

A miscalibrated model cannot support a cost-based approval threshold, so calibration is a
precondition of notebook 06 rather than a constraint the ladder imposes. It is applied to R0
here on the same terms it will be applied to the challenger: isotonic regression fitted on the
**calibration split**, which neither the binning nor the logistic regression has seen.

In [ ]:
isotonic = IsotonicRegression(out_of_bounds="clip")
isotonic.fit(probability["calibration"], y[is_calibration])
calibrated_test = isotonic.predict(probability["test"])

raw = evaluation.metric_set(y[is_test], probability["test"], split="test")
calibrated = evaluation.metric_set(y[is_test], calibrated_test, split="test")

comparison = pd.DataFrame([raw, calibrated], index=["R0 raw", "R0 isotonic"])
comparison[["pr_auc", "auc", "ks", "gini", "brier", "ece"]].round(6)

In [ ]:
bins = calibration_bins(y[is_test], probability["test"], n_bins=10)
bins.round(5)

**A negative result, reported as one.** Isotonic regression cuts expected calibration error
roughly in half, leaves the Brier score fractionally *worse*, and costs a little
discrimination.

That last part is worth being precise about, because the obvious claim is wrong. An isotonic
map is **non-decreasing, not strictly increasing**: it cannot reverse any pair, so no
applicant overtakes a safer one, but it maps ranges of distinct probabilities onto a single
fitted value. Those ties are indistinguishable to a ranking metric, so AUC and PR-AUC fall
slightly rather than staying identical. PR-AUC falls further than AUC because average
precision is sensitive to ties near the top of the ranking, which is exactly where isotonic's
steps are widest at an 8% base rate.

The calibration gain is real and the discrimination cost is real, and neither cancels the
other — they are different properties, which is the reason both are in the table.

The rest is what should happen. A logistic regression on weight-of-evidence-transformed
inputs is calibrated by construction: fitted by maximum likelihood on the log-odds scale, its
probabilities already mean what they say. The scorecard's raw ECE is around two parts in a
thousand; there is very little for isotonic to correct, and what it does correct it pays for
in a coarser probability.

The consequence for the ladder is that R2 is nearly free for R0. Whether it is free for the
gradient booster is an open question, and notebook 03 answers it. If calibration turns out to
cost the challenger something the scorecard never had to pay, that is part of the answer to
whether the accuracy gain survives.

## 9. Persist

In [ ]:
# The fitted scorecard is persisted so notebook 04 can compute its reason codes on
# the same terms as the challenger's. For a linear model on WoE-transformed inputs the
# SHAP value of a characteristic is exactly its coefficient times its centred WoE, so
# the two are directly comparable rather than merely analogous — but that needs the
# fitted binning process, which cannot be reconstructed from the points table alone.
with (processed / "r0_scorecard.pkl").open("wb") as handle:
    pickle.dump({"scorecard": card, "characteristics": chosen, "isotonic": isotonic}, handle)
print(f"wrote r0_scorecard.pkl ({(processed / 'r0_scorecard.pkl').stat().st_size / 1e6:.1f} MB)")

In [ ]:
# Scattered back by position rather than concatenated, so an applicant's prediction
# cannot silently misalign with their row if the split order ever changes.
ordered_probability = np.empty(len(frame), dtype=float)
ordered_score = np.empty(len(frame), dtype=float)
for name, mask in [("train", is_train), ("calibration", is_calibration), ("test", is_test)]:
    ordered_probability[mask] = probability[name]
    ordered_score[mask] = score[name]

predictions = pd.DataFrame(
    {
        "SK_ID_CURR": frame["SK_ID_CURR"].values,
        "split": splits,
        "TARGET": y,
        "prob": ordered_probability,
        "prob_calibrated": isotonic.predict(ordered_probability),
        "points": ordered_score,
    }
)
assert np.allclose(predictions.loc[is_test, "prob"].values, probability["test"])
assert np.allclose(predictions.loc[is_train, "prob"].values, probability["train"])

# optbinning's Bin column mixes interval arrays with categorical level lists, which
# parquet cannot type. Rendered to string on the way out; it is a label, not a value.
points_out = points.copy()
points_out["Bin"] = points_out["Bin"].astype(str)

predictions.to_parquet(processed / "r0_predictions.parquet", index=False)
selection_log.to_parquet(processed / "r0_selection_log.parquet", index=False)
points_out.to_parquet(processed / "r0_points_table.parquet", index=False)
print(f"wrote r0_predictions.parquet ({len(predictions):,} rows)")
print(f"wrote r0_points_table.parquet ({len(points_out):,} scored bins)")

In [ ]:
r0 = {
    "rung": "R0",
    "description": "weight-of-evidence scorecard, logistic regression, PDO points scaling",
    "seed": sp.SEED,
    "n_candidates": len(candidates),
    "n_above_iv_floor": len(above_floor),
    "n_characteristics": len(chosen),
    "characteristics": chosen,
    "iv_floor": sc.IV_FLOOR,
    "max_correlation": sc.MAX_CORRELATION,
    "scaling": {"pdo": sc.PDO, "base_odds": sc.BASE_ODDS, "base_points": sc.BASE_POINTS},
    "metrics_by_split": {row["split"]: row for row in rows},
    "test_intervals": {k: {"low": lo, "high": hi} for k, (lo, hi) in intervals.items()},
    "calibrated_test": calibrated,
    "verification": {
        "woe_vs_optbinning_max_abs_diff": float(checks["max |d WoE|"].max()),
        "iv_vs_optbinning_max_abs_diff": float(checks["|d IV|"].max()),
        "german_credit_hand_check_abs_diff": float(abs(by_hand - by_code)),
        "points_scaling_slope": float(slope),
        "points_scaling_expected_slope": float(expected_factor),
        "points_scaling_max_residual": residual,
    },
}
(paths.metrics_dir() / "r0.json").write_text(json.dumps(r0, indent=2, default=float) + "\n")
print("wrote metrics/r0.json")

In [ ]:
auc_gap = float(metrics_by_split.loc["train", "auc"] - metrics_by_split.loc["test", "auc"])
selected_rows = selection_log.loc[selection_log["selected"]]
dropped_correlation = int(selection_log["reason"].str.startswith("correlates").sum())
top_points = points[["Variable", "Bin", "Count", "Event rate", "WoE", "Points"]]

lines = [
    "# ADIL — R0, the weight-of-evidence scorecard",
    "",
    "Generated by `notebooks/02_scorecard.ipynb`. Every number is read from a fitted model",
    "or a data artifact, not typed.",
    "",
    "MAIB AI 217 · SP Jain School of Global Management, Dubai · Krishna Mathur",
    "",
    "## What this is",
    "",
    "Rung R0 of the constraint ladder — the baseline the gradient booster must beat. A weak",
    "baseline is a form of dishonesty, so this one gets monotonic optimal binning,",
    "information-value selection with correlation pruning, and industry-standard points",
    "scaling.",
    "",
    f"- Candidates considered: **{len(candidates)}** ({len(categorical)} categorical)",
    f"- Above the IV floor of {sc.IV_FLOOR}: **{len(above_floor)}**",
    f"- Characteristics in the scorecard: **{len(chosen)}**",
    f"- Dropped for correlation above {sc.MAX_CORRELATION}: **{dropped_correlation}**",
    "",
    "Sex and age are excluded from the candidate set. Deciding on them would be disparate",
    "treatment; they are retained in the frame for measurement only.",
    "",
    "## Verification",
    "",
    "Three checks, none of which is optbinning checking itself.",
    "",
    "**ADIL's arithmetic against optbinning's.** `adil.scorecard` implements weight of",
    "evidence and information value from the formula, independently of the library.",
    "Compared on three characteristics including a categorical one:",
    "",
    "| Characteristic | Bins | max abs diff WoE | abs diff IV |",
    "|---|---:|---:|---:|",
]
for check in checks.to_dict("records"):
    lines.append(
        f"| `{check['characteristic']}` | {check['bins']} | "
        f"{check['max |d WoE|']:.2e} | {check['|d IV|']:.2e} |"
    )
lines += [
    "",
    "Agreement to machine precision. optbinning's `Missing` bin is a real bin carrying a",
    "fifth of the rows for `EXT_SOURCE_3` and counts towards the totals; verifying on the",
    "non-missing bins alone changes the denominators and produces a mismatch that looks like",
    "a library bug and is not.",
    "",
    "Empty bins differ by convention, not arithmetic: optbinning reports WoE `0.0`, ADIL",
    "reports `NaN`. Zero claims the bin carries no evidence; NaN claims the sample cannot",
    "answer for that bin. Both contribute nothing to the IV total.",
    "",
    "**Against a hand computation.** German Credit, 1,000 rows, 700 good and 300 bad.",
    f"Attribute 6 level A65 (no savings account) has {a65_event} bad of {a65_n}:",
    "",
    f"    ln(({a65_nonevent}/{int(total_nonevent)}) / "
    f"({a65_event}/{int(total_event)})) = {by_hand:.6f}",
    "",
    f"ADIL computes {by_code:.6f}; the two agree to {abs(by_hand - by_code):.1e}. This is the",
    "only use German Credit is put to here — different target, different country, it cannot",
    "baseline anything on Home Credit. It is present because 1,000 rows can be counted.",
    "",
    "**Points scaling is affine in the log-odds.** Fitting score against log-odds recovers",
    f"a slope of {slope:.6f} against the expected `PDO/ln 2` = {expected_factor:.6f}, with",
    f"maximum residual {residual:.1e}. Scaling therefore changes what a score looks like and",
    "changes nothing about ranking, probability or decision, which is why every metric below",
    "is computed on the probability rather than the points.",
    "",
    "## Discrimination and calibration",
    "",
    "PR-AUC leads: at an 8% base rate AUC flatters, because a model can rank well overall",
    "and still be uninformative where the approval cutoff sits. Gini is reported because a",
    "credit audience reads it.",
    "",
    "| Split | n | PR-AUC | AUC | KS | Gini | Brier | ECE |",
    "|---|---:|---:|---:|---:|---:|---:|---:|",
]
for name, row in metrics_by_split.iterrows():
    lines.append(
        f"| {name} | {int(row['n']):,} | {row['pr_auc']:.4f} | {row['auc']:.4f} | "
        f"{row['ks']:.4f} | {row['gini']:.4f} | {row['brier']:.5f} | {row['ece']:.5f} |"
    )
lines += [
    "",
    f"The train-to-test AUC gap is {auc_gap:.4f},",
    "so the baseline is not overfitted and the challenger is being asked to beat a real one.",
    "",
    "### Test-set intervals",
    "",
    "Percentile bootstrap over resampled predictions. This is how precisely the metric is",
    "known **on this test set**; it is not how much the metric would move on different",
    "training data, which is larger and is not estimated in this project.",
    "",
    "| Metric | Estimate | 2.5% | 97.5% |",
    "|---|---:|---:|---:|",
]
for key, (low, high) in intervals.items():
    lines.append(f"| {key} | {float(test_metrics[key]):.5f} | {low:.5f} | {high:.5f} |")
lines += [
    "",
    "## Calibration — a negative result",
    "",
    "Isotonic regression fitted on the calibration split, which neither the binning nor the",
    "logistic regression has seen.",
    "",
    "| | PR-AUC | AUC | Brier | ECE |",
    "|---|---:|---:|---:|---:|",
    f"| R0 raw | {raw['pr_auc']:.5f} | {raw['auc']:.5f} | {raw['brier']:.6f} | {raw['ece']:.6f} |",
    f"| R0 isotonic | {calibrated['pr_auc']:.5f} | {calibrated['auc']:.5f} | "
    f"{calibrated['brier']:.6f} | {calibrated['ece']:.6f} |",
    "",
    f"Calibration cuts ECE from {raw['ece']:.6f} to {calibrated['ece']:.6f}, leaves Brier",
    f"fractionally worse ({raw['brier']:.6f} to {calibrated['brier']:.6f}), and costs a",
    f"little discrimination: AUC {raw['auc']:.5f} to {calibrated['auc']:.5f}, PR-AUC",
    f"{raw['pr_auc']:.5f} to {calibrated['pr_auc']:.5f}.",
    "",
    "The discrimination cost is worth being precise about, because the obvious claim — that",
    "a monotone recalibration leaves ranking untouched — is wrong. An isotonic map is",
    "*non-decreasing*, not strictly increasing. It cannot reverse a pair, so no applicant",
    "overtakes a safer one, but it collapses ranges of distinct probabilities onto a single",
    "fitted value. Those ties are indistinguishable to a ranking metric, so AUC and PR-AUC",
    "fall slightly. PR-AUC falls further because average precision is sensitive to ties near",
    "the top of the ranking, which is where isotonic's steps are widest at an 8% base rate.",
    "",
    "The calibration gain and the discrimination cost are both real and neither cancels the",
    "other; they are different properties, which is why both are in the table.",
    "",
    "The rest is the expected outcome, reported rather than tidied away. A logistic",
    "regression on weight-of-evidence inputs is calibrated by construction — fitted by",
    "maximum likelihood on the log-odds scale, its probabilities already mean what they say.",
    "There is very little for isotonic to correct and what it corrects it pays for in a",
    "coarser probability.",
    "",
    "The consequence for the ladder: rung R2 is nearly free for R0. Whether it is free for",
    "the gradient booster is open, and notebook 03 answers it. If calibration costs the",
    "challenger something the scorecard never had to pay, that is part of the answer to",
    "whether the accuracy gain survives.",
    "",
    "## Characteristics",
    "",
    "| # | Characteristic | IV |",
    "|---:|---|---:|",
]
for i, row in enumerate(selected_rows.itertuples(), start=1):
    lines.append(f"| {i} | `{row.feature}` | {row.iv:.4f} |")
lines += [
    "",
    "### Rejected candidates, with reasons",
    "",
    "| Characteristic | IV | Reason |",
    "|---|---:|---|",
]
for row in selection_log.loc[~selection_log["selected"]].head(25).itertuples():
    lines.append(f"| `{row.feature}` | {row.iv:.4f} | {row.reason} |")
lines += [
    "",
    f"({int((~selection_log['selected']).sum())} rejected in total; the first 25 are shown.",
    "The full log is in `r0_selection_log.parquet`.)",
    "",
    "## Limitations",
    "",
    "- Binning is fitted on the training split only, so bin edges do not see the test target.",
    "  Selection is likewise made on training information values.",
    "- The scorecard is fitted once on one split. Metric stability across resamples of the",
    "  *training* data is not estimated; the intervals above cover measurement on the test",
    "  set only.",
    "- Weight of evidence for a level such as German Credit's A65 reports what the sample",
    "  says, not what credit policy should do about it. A counterintuitive bin is a prompt",
    "  for challenge, not a finding.",
    "",
]
path = paths.reports_dir() / "scorecard.md"
path.write_text("\n".join(lines) + "\n")
print(f"wrote {path} ({len(lines)} lines)")